Реализуйте алгоритм SAC для среды lunar lander

In [ ]:
!pip install swig
!pip install "gymnasium[box2d]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 17.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.4/374.4 kB 7.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [23]:
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from collections import deque
import random
from torch.distributions import Normal

In [24]:
GAMMA = 0.99
TAU = 0.005
ALPHA = 0.2
ACTOR_LR = 3e-4
CRITIC_LR = 3e-4
REPLAY_SIZE = 100000
BATCH_SIZE = 256
START_STEPS = 10000
TOTAL_STEPS = 200000
UPDATE_AFTER = 1000
UPDATE_EVERY = 50

In [25]:
class Actor(nn.Module):
    def __init__(self, obs_dim, act_dim, action_low, action_high):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(),
        )
        self.mu_layer = nn.Linear(256, act_dim)
        self.log_std_layer = nn.Linear(256, act_dim)
        self.action_low = action_low
        self.action_high = action_high

    def forward(self, obs):
        x = F.relu(self.net(obs))
        mean, std = self.mu_layer(x),  torch.clamp(self.log_std_layer(x), -20, 2).exp()
        normal = torch.distributions.Normal(mean, std)

        x_t = normal.rsample()
        y_t = torch.tanh(x_t)
        action = y_t * (action_high - action_low) / 2.0 + (action_low + action_high) / 2.0

        log_prob = normal.log_prob(x_t)
        log_prob -= torch.log((1 - y_t.pow(2)) + 1e-6)
        log_prob = log_prob.sum(1, keepdim=True)

        return action, log_prob

    def get_action(self, obs, deterministic=False):
      with torch.no_grad():
            if deterministic:
                x = F.relu(self.net(obs))
                mean = self.mu_layer(x)
                action = torch.tanh(mean) * (self.action_high - self.action_low) / 2 + (self.action_high + self.action_low) / 2
            else:
                action, _ = self.forward(obs)
            return action.squeeze(0)

In [26]:
class Critic(nn.Module):
    def __init__(self, obs_dim, act_dim):
        super().__init__()
        self.q1 = nn.Sequential(
            nn.Linear(obs_dim + act_dim, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(),
            nn.Linear(256, 1)
        )
        self.q2 = nn.Sequential(
            nn.Linear(obs_dim + act_dim, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(),
            nn.Linear(256, 1)
        )

    def forward(self, obs, act):
        x = torch.cat([obs, act], dim=-1)
        return self.q1(x), self.q2(x)

In [27]:
class ReplayBuffer:
    def __init__(self, size):
        self.buffer = deque(maxlen=size)

    def add(self, *args):
        self.buffer.append(tuple(args))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = map(np.array, zip(*batch))
        return (
            torch.tensor(states, dtype=torch.float32),
            torch.tensor(actions, dtype=torch.float32),
            torch.tensor(rewards, dtype=torch.float32).unsqueeze(1),
            torch.tensor(next_states, dtype=torch.float32),
            torch.tensor(dones, dtype=torch.float32).unsqueeze(1)
        )

In [28]:
env = gym.make("LunarLanderContinuous-v3")
obs_dim = env.observation_space.shape[0]
act_dim = env.action_space.shape[0]
action_low, action_high = float(env.action_space.low[0]), float(env.action_space.high[0])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
actor = Actor(obs_dim, act_dim, action_low, action_high).to(device)
critic = Critic(obs_dim, act_dim).to(device)
critic_target = Critic(obs_dim, act_dim).to(device)
critic_target.load_state_dict(critic.state_dict())

actor_opt = optim.Adam(actor.parameters(), lr=ACTOR_LR)
critic_opt = optim.Adam(critic.parameters(), lr=CRITIC_LR)

replay = ReplayBuffer(REPLAY_SIZE)

obs, _ = env.reset()
episode_return, episode_len = 0, 0

In [29]:
def update():
    states, actions, rewards, next_states, dones = replay.sample(BATCH_SIZE)
    states = states.to(device)
    actions = actions.to(device)
    rewards = rewards.to(device)
    next_states = next_states.to(device)
    dones = dones.to(device)

    with torch.no_grad():
        next_actions, next_log_probs = actor(next_states)
        q1_next, q2_next = critic_target(next_states, next_actions)
        q_next = torch.min(q1_next, q2_next) - ALPHA * next_log_probs
        target_q = rewards + GAMMA * (1 - dones) * q_next

    q1, q2 = critic(states, actions)
    critic_loss = F.mse_loss(q1, target_q) + F.mse_loss(q2, target_q)

    critic_opt.zero_grad()
    critic_loss.backward()
    critic_opt.step()

    actions_pred, log_probs = actor(states)
    q1_actor, q2_actor = critic(states, actions_pred)
    actor_loss = (ALPHA * log_probs - torch.min(q1_actor, q2_actor)).mean()

    actor_opt.zero_grad()
    actor_loss.backward()
    actor_opt.step()

    with torch.no_grad():
        for param, target_param in zip(critic.parameters(), critic_target.parameters()):
            target_param.data.copy_(TAU * param.data + (1 - TAU) * target_param.data)

In [30]:
for step in range(TOTAL_STEPS):
    if step < START_STEPS:
        act = env.action_space.sample()
    else:
        with torch.no_grad():
            obs_t = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
            act = actor.get_action(obs_t).cpu().numpy()

    next_obs, rew, terminated, truncated, _ = env.step(act)
    done = terminated or truncated
    replay.add(obs, act, rew, next_obs, done)

    obs = next_obs
    episode_return += rew
    episode_len += 1

    if done:
        obs, _ = env.reset()
        print(f"Step: {step}, Return: {episode_return:.2f}, Len: {episode_len}")
        episode_return, episode_len = 0, 0

    if step >= UPDATE_AFTER and step % UPDATE_EVERY == 0:
        for _ in range(UPDATE_EVERY):
            update()

Step: 113, Return: -263.04, Len: 114
Step: 218, Return: -274.74, Len: 105
Step: 332, Return: -301.39, Len: 114
Step: 407, Return: -64.13, Len: 75
Step: 586, Return: -126.58, Len: 179
Step: 696, Return: -311.40, Len: 110
Step: 797, Return: -377.73, Len: 101
Step: 921, Return: -150.65, Len: 124
Step: 1012, Return: -78.32, Len: 91
Step: 1073, Return: -68.27, Len: 61
Step: 1238, Return: -365.13, Len: 165
Step: 1335, Return: -184.61, Len: 97
Step: 1490, Return: -418.21, Len: 155
Step: 1620, Return: -234.73, Len: 130
Step: 1704, Return: -117.65, Len: 84
Step: 1796, Return: -56.03, Len: 92
Step: 1867, Return: -103.68, Len: 71
Step: 1961, Return: -125.76, Len: 94
Step: 2190, Return: 12.97, Len: 229
Step: 2257, Return: -138.94, Len: 67
Step: 2363, Return: -265.81, Len: 106
Step: 2519, Return: -171.58, Len: 156
Step: 2613, Return: -77.76, Len: 94
Step: 2718, Return: -294.52, Len: 105
Step: 2800, Return: -65.05, Len: 82
Step: 2888, Return: -471.84, Len: 88
Step: 3062, Return: -75.12, Len: 174
Ste